# Data Engineering, SQL, and Reproducibility

This notebook is intentionally practical: data quality, SQL, transactions, and reproducibility are all taught as operational habits rather than isolated theory.


## Tables and keys

A table is a collection of related records. Keys help us connect tables while maintaining data integrity.


In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON')
conn.execute('CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, region TEXT NOT NULL)')
conn.execute('CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL, amount REAL NOT NULL, FOREIGN KEY(customer_id) REFERENCES customers(customer_id))')
conn.executemany('INSERT INTO customers VALUES (?, ?)', [(1, 'North'), (2, 'South')])
conn.executemany('INSERT INTO orders VALUES (?, ?, ?)', [(101, 1, 25.0), (102, 1, 15.0), (103, 2, 40.0)])
print(conn.execute('SELECT COUNT(*) FROM orders').fetchone())
assert conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0] == 3


## Join and summarize

Once students know how to create tables, they should immediately learn how to join them and summarize a business question.


In [ ]:
query = '''
SELECT customers.region, SUM(orders.amount) as total_amount
FROM customers
JOIN orders ON orders.customer_id = customers.customer_id
GROUP BY customers.region
ORDER BY total_amount DESC
'''
print(conn.execute(query).fetchall())
assert round(sum(amount for _, amount in conn.execute(query).fetchall()), 2) == 80.0


## Transactions and validation

A transaction is a unit of work: either it fully commits or it rolls back. This is a very important concept in production data work.


In [ ]:
before = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
conn.execute('SAVEPOINT bad_order')
try:
    conn.execute('INSERT INTO orders VALUES (?, ?, ?)', (104, 999, 12.0))
    conn.commit()
except sqlite3.IntegrityError:
    conn.execute('ROLLBACK TO SAVEPOINT bad_order')
    conn.execute('RELEASE bad_order')
print(conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0])
assert conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0] == before
assert before == 3


## Reproducible workflow

Data engineering is not only SQL. It is also tracking assumptions, writing clean transformations, and knowing which checks must happen before a pipeline can be trusted.


In [ ]:
def region_summary(db):
    rows = db.execute('SELECT region, SUM(amount) FROM (SELECT customers.region, orders.amount FROM customers JOIN orders ON orders.customer_id = customers.customer_id) GROUP BY region').fetchall()
    return rows
print(region_summary(conn))
assert isinstance(region_summary(conn), list)
